# LSM-Trees Internals: MemTable, SSTables, Bloom Filters & Leveled Compaction

Interactive hands-on sandbox exploring internal storage mechanics, algorithmic data structures, and architectural invariants.


In [ ]:
import sys
from pathlib import Path

# Prepend project_solution to sys.path so Track A internal engines import cleanly
ps_dir = Path('.').resolve() / 'project_solution'
if str(ps_dir) not in sys.path:
    sys.path.insert(0, str(ps_dir))

print(f'Python runtime: {sys.version.split()[0]}')
print('Loaded internal mechanics for: Module_15_LSM_Trees_Compaction_DynamoDB')


## 1. Engine Initialization & Setup

Importing the module's Track A internal simulation engine and instantiating state.


In [ ]:
from lsm_dynamo_engine import BloomFilter, MemTable

# Initialize MemTable with maximum entries threshold
memtable = MemTable(max_entries=3)
memtable.put("user:charlie", {"name": "Charlie"}, timestamp_us=100)
memtable.put("user:alice", {"name": "Alice"}, timestamp_us=101)
memtable.put("user:bob", {"name": "Bob"}, timestamp_us=102)

print(f"MemTable full status: {memtable.is_full()} (Storage count: {len(memtable.storage)})")


## 2. Core Architectural Operations & State Mutation

Executing data mutations, transactions, or indexing procedures.


In [ ]:
# MemTable Flush to Disk as Sorted String Table (SSTable)
sstable = memtable.flush()
print(f"Flushed to SSTable. Keys strictly sorted: {[e[0] for e in sstable.entries]}")
print(f"MemTable storage count after flush: {len(memtable.storage)}")


## 3. Performance Micro-Benchmarking & Invariant Verification

Evaluating execution latency, cache hits, or computational trade-offs.


In [ ]:
# Bloom Filter Fast Negative Rejection:
# Before scanning SSTable files on disk, Bloom Filter rejects non-existent keys in O(1) RAM!
bf = BloomFilter(expected_items=50, false_positive_rate=0.01)
bf.add("user:alice")
bf.add("user:bob")

print("Key 'user:alice' in Bloom Filter:", bf.contains("user:alice"))
print("Key 'user:non_existent' in Bloom Filter:", bf.contains("user:non_existent"))


## 4. Architectural Invariant Verification

Asserting mathematical correctness and durability invariants.


In [ ]:
# Verify LSM Invariants
assert bf.contains("user:alice") is True
assert bf.contains("user:non_existent") is False
assert [e[0] for e in sstable.entries] == ["user:alice", "user:bob", "user:charlie"]
print("[+] LSM-Tree MemTable, SSTable, and Bloom Filter invariants verified successfully!")


## Summary & Operational DBRE Best Practices

1. **Never bypass serialization contracts:** Always enforce binary-safe schemas and validated boundaries.
2. **Monitor buffer and memory allocations:** Understand the latency cliff when in-memory structures spill to disk.
3. **Ensure idempotency across replication tiers:** Distributed operations must survive retries without corrupting state.
